# 01. Exploratory Data Analysis & Data Quality Investigation

**Project**: CineMatch — TMDB Content-Based Movie Recommendation System  
**Dataset**: TMDB 5000 Movies & Credits Dataset  
**Methodology**: Faithful reproduction & modernization of CampusX content-based recommendation approach  

### Objectives:
1. Inspect dataset dimensions, feature schemas, and missing value patterns.
2. Analyze title homonyms and potential data corruption during merges.
3. Investigate distributions of genres, release years, vote averages, and overview word counts.
4. Document preprocessing implications for feature engineering.

In [1]:
import os
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
print('Libraries successfully imported.')

Libraries successfully imported.


## 1. Load Raw Datasets

In [2]:
movies_path = '../data/raw/tmdb_5000_movies.csv'
credits_path = '../data/raw/tmdb_5000_credits.csv'

# Fallback if running from root
if not os.path.exists(movies_path):
    movies_path = 'data/raw/tmdb_5000_movies.csv'
    credits_path = 'data/raw/tmdb_5000_credits.csv'

movies = pd.read_csv(movies_path)
credits = pd.read_csv(credits_path)

print(f'Movies table shape: {movies.shape}')
print(f'Credits table shape: {credits.shape}')

Movies table shape: (4803, 20)
Credits table shape: (4803, 4)


In [3]:
movies.head(3)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466


In [4]:
credits.head(3)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


## 2. Missing Value & Null Pattern Analysis

In [5]:
null_movies = movies.isnull().sum()
null_movies = null_movies[null_movies > 0].to_frame('missing_count')
null_movies['percentage'] = (null_movies['missing_count'] / len(movies)) * 100
null_movies.sort_values('missing_count', ascending=False)

,missing_count,percentage
homepage,3091,64.355611
tagline,844,17.572351
overview,3,0.062461
runtime,2,0.041641
release_date,1,0.020820


In [6]:
null_credits = credits.isnull().sum()
print('Credits missing values:')
print(null_credits)

Credits missing values:
movie_id    0
title       0
cast        0
crew        0
dtype: int64


## 3. Investigating Duplicate Titles & Merge Safety

In [7]:
# Find duplicate movie titles
dup_titles = movies[movies.duplicated(subset=['title'], keep=False)]
dup_titles[['id', 'title', 'release_date', 'vote_average']].sort_values('title')

,id,title,release_date,vote_average
1359,268,Batman,1989-06-23,7.0
4267,2661,Batman,1966-07-30,6.1
3647,39269,Out of the Blue,1980-05-01,6.5
3693,10844,Out of the Blue,2006-10-12,5.9
972,72710,The Host,2013-03-22,6.0
2877,1255,The Host,2006-07-27,6.7


In [8]:
# Compare merge strategies: merge on 'title' vs merge on 'id' == 'movie_id'
merge_title = movies.merge(credits, on='title')
merge_id = movies.merge(credits, left_on='id', right_on='movie_id')

print(f'Rows after merge on title: {merge_title.shape[0]} (Cartesian product on duplicates)')
print(f'Rows after merge on id: {merge_id.shape[0]} (Exact 1:1 join preserved)')

Rows after merge on title: 4809 (Cartesian product on duplicates)
Rows after merge on id: 4803 (Exact 1:1 join preserved)


## 4. Genre Frequency Analysis

In [9]:
def extract_genres(raw):
    try:
        data = ast.literal_eval(raw)
        return [item['name'] for item in data if 'name' in item]
    except Exception:
        return []

movies['genre_list'] = movies['genres'].apply(extract_genres)
all_genres = [g for sublist in movies['genre_list'] for g in sublist]
genre_counts = pd.Series(all_genres).value_counts()
print(f'Total distinct genres: {len(genre_counts)}')
print('Top 10 Genres:\n', genre_counts.head(10))

Total distinct genres: 20
Top 10 Genres:
 Drama              2297
Comedy             1722
Thriller           1274
Action             1154
Romance             894
Adventure           790
Crime               696
Science Fiction     535
Horror              519
Family              513
Name: count, dtype: int64


## 5. Overview Word Length Distribution

In [10]:
movies['overview_words'] = movies['overview'].fillna('').apply(lambda x: len(x.split()))
print(movies['overview_words'].describe())

count    4803.000000
mean       52.124714
std        27.888777
min         0.000000
25%        30.000000
50%        48.000000
75%        67.000000
max       175.000000
Name: overview_words, dtype: float64

## 6. Release Year & Rating Distributions

In [11]:
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')
movies['release_year'] = movies['release_date'].dt.year
print(f'Release year range: {movies["release_year"].min()} to {movies["release_year"].max()}')
print(f'Vote average mean: {movies["vote_average"].mean():.2f}, std: {movies["vote_average"].std():.2f}')

Release year range: 1916.0 to 2017.0
Vote average mean: 6.09, std: 1.19


## 7. Conclusions & Findings for Feature Engineering

1. **Merge Protocol**: Merging on `id == movie_id` avoids contaminating homonymous titles.
2. **Null Values**: Exactly 3 movies lack `overview`. Dropping or imputing blank string preserves 4,803 or 4,806 movies.
3. **Structured Fields**: Genres, keywords, cast, and crew require safe JSON parsing via `ast.literal_eval`.
4. **Entity Collapse**: Multi-word entity names must have whitespace removed to prevent false token associations.